In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


def _edge_records(graph):
    from topologicpy.TGraph import TGraph
    records = TGraph.Edges(graph)
    if any(record.get("representation") is None for record in records):
        geometry = TGraph.Edges(graph, asTopologic=True, segmentCurves=False)
        if len(geometry) != len(records):
            raise ValueError("Could not generate one geometry representation per graph edge")
        for record, edge in zip(records, geometry):
            record["representation"] = edge
    return records


In [ ]:
from topologicpy.Topology import Topology
from topologicpy.TGraph import TGraph
from topologicpy.Helper import Helper

print(Helper.Version())

## GraphRAG Generation POC-3: Claude API Integration - Suggesting Nodes and Edges Iteratively with TopologicPy

### STEP 1: Supply your API Key

In [ ]:
import os

# Check for API key based on selected backend
backend = os.environ.get("LLM_BACKEND", "anthropic").lower()

if backend == "anthropic":
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        api_key = input("Please provide your Anthropic API Key:")
else:
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        api_key = input("Please provide your OpenAI API Key:")

### Step 2: Supply the number of bedrooms and bathrooms
#### Type something like: "three bedrooms and two bathrooms"

In [ ]:
user_prompt = input("How many bedrooms and bathrooms should it have?")
if user_prompt == None:
    user_prompt = "three bedrooms and two bathrooms"

### STEP 3: Supply a folder location to save the PNG images

In [ ]:
folder = input("Please provide a folder path to save the PNG images:")

### STEP 4: Main Generative AI code with Multi-Backend Support (Including Claude)

In [ ]:
# =========================
# TopologicPy + GraphRAG + LLM (with area support and Claude integration)
# =========================

# -- TopologicPy imports
from topologicpy.TGraph import TGraph
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Dictionary import Dictionary
from topologicpy.Topology import Topology

# -- Standard + ML / embeddings
import os, re, math
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

import random

# --------------------------------------------------------------------
# Embedding model for KB (GraphRAG-ish retrieval) - Fixed corruption issue
# --------------------------------------------------------------------
# Use explicit variable to prevent string corruption
MODEL_NAME = "all-MiniLM-L6-v2"

try:
    print(f"Loading SentenceTransformer model: {MODEL_NAME}")
    model = SentenceTransformer(MODEL_NAME)
    model.tokenizer.clean_up_tokenization_spaces = True  # keep behavior consistent
    print("✅ SentenceTransformer model loaded successfully")
except Exception as e:
    print(f"❌ Failed to load {MODEL_NAME}: {e}")
    # Fallback options
    fallback_models = [
        "paraphrase-MiniLM-L6-v2",
        "all-mpnet-base-v2", 
        "distilbert-base-nli-mean-tokens"
    ]
    
    model = None
    for fallback in fallback_models:
        try:
            print(f"Trying fallback model: {fallback}")
            model = SentenceTransformer(fallback)
            model.tokenizer.clean_up_tokenization_spaces = True
            print(f"✅ Successfully loaded fallback model: {fallback}")
            break
        except Exception as fe:
            print(f"❌ Fallback {fallback} failed: {fe}")
            continue
    
    if model is None:
        raise RuntimeError("Failed to load any SentenceTransformer model. Please check your internet connection and try again.")

# --------------------------------------------------------------------
# LLM backend (Multi-backend: OpenAI, Anthropic, Ollama, LM Studio)
# --------------------------------------------------------------------

# Backend options:
# "openai"     -> Official OpenAI (needs OPENAI_API_KEY)
# "anthropic"  -> Anthropic Claude (needs ANTHROPIC_API_KEY)  
# "ollama"     -> Local Ollama (http://localhost:11434/v1)
# "lmstudio"   -> Local LM Studio (http://localhost:1234/v1)
BACKEND = os.getenv("LLM_BACKEND", "anthropic").lower()

# Choose a default model per backend; override via env LLM_MODEL if you like
LLM_MODEL = os.getenv(
    "LLM_MODEL",
    "claude-3-5-sonnet-20241022" if BACKEND == "anthropic" else
    "gpt-4o-mini" if BACKEND == "openai" else
    "llama3.1" if BACKEND == "ollama" else
    "local-model"
)

def make_client():
    if BACKEND == "anthropic":
        import anthropic
        return anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", api_key))
    elif BACKEND == "openai":
        from openai import OpenAI
        return OpenAI(api_key=os.environ.get("OPENAI_API_KEY", api_key))
    elif BACKEND == "ollama":
        from openai import OpenAI
        base_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1")
        return OpenAI(base_url=base_url, api_key=os.environ.get("OLLAMA_API_KEY", "ollama"))
    elif BACKEND == "lmstudio":
        from openai import OpenAI
        base_url = os.environ.get("LMSTUDIO_BASE_URL", "http://localhost:1234/v1")
        return OpenAI(base_url=base_url, api_key=os.environ.get("LMSTUDIO_API_KEY", "lmstudio"))
    else:
        raise ValueError(f"Unknown LLM backend: {BACKEND}")

client = make_client()
print(f"Using {BACKEND} backend with model: {LLM_MODEL}")

# =========================
# Knowledge base (GraphRAG)
# =========================

# A small, typed adjacency corpus + rules of thumb about typical room adjacencies and areas (m²).
# You can extend/replace freely; these strings are embedded and retrieved.
known_graphs = [
    # name, list of (room_type, [adjacent_types])
    ("basic_flat",
     [("living room", ["dining room", "kitchen", "hallway"]),
      ("kitchen", ["dining room", "living room"]),
      ("bedroom", ["hallway", "bathroom"]),
      ("bathroom", ["hallway"]),
      ("dining room", ["living room", "kitchen"])]),
    ("compact_studio",
     [("studio", ["kitchen", "bathroom"]),
      ("kitchen", ["studio"]),
      ("bathroom", ["studio"])]),
    ("two_bed",
     [("living room", ["dining room", "kitchen", "hallway"]),
      ("kitchen", ["dining room", "living room"]),
      ("bedroom", ["hallway", "bathroom"]),
      ("bedroom", ["hallway"]),
      ("bathroom", ["hallway"]),
      ("dining room", ["living room", "kitchen"]),
      ("hallway", ["living room", "bedroom", "bathroom"])]),
]

# Turn graph patterns + heuristics into retrievable text snippets
def _pattern_to_text(name, pairs):
    lines = [f"Pattern: {name}."]
    for t, neigh in pairs:
        if neigh:
            lines.append(f"Room '{t}' is commonly connected to {', '.join([repr(n) for n in neigh])}.")
        else:
            lines.append(f"Room '{t}' may be isolated.")
    return " ".join(lines)

# Heuristic area priors (very rough defaults; tweak for your domain)
area_hints = {
    "living room": (18, 35),
    "dining room": (10, 20),
    "kitchen": (10, 16),
    "bedroom": (10, 16),
    "bathroom": (4, 8),
    "hallway": (4, 10),
    "studio": (20, 35),
    "storage": (2, 6),
    "pantry": (2, 5),
    "office": (8, 14),
    "balcony": (3, 8),
}

def _area_hint_for(room_type):
    rt = room_type.strip().lower()
    lo, hi = area_hints.get(rt, (8, 18))
    # pick a mid or rounded suggestion for examples
    return round((lo + hi) / 2.0, 1)

kb_prompts = []
for name, pairs in known_graphs:
    kb_prompts.append(_pattern_to_text(name, pairs))

# Optional: add some "rule" snippets that LLM can see via retrieval
kb_prompts.extend([
    "Kitchens are typically adjacent to dining rooms and often near living rooms.",
    "Bedrooms are usually off a hallway and near bathrooms; bathrooms connect to hallway rather than living spaces.",
    "In compact layouts, consider adding a hallway to mediate connections among bedrooms and bathrooms.",
    "Dining rooms commonly bridge kitchens and living rooms.",
])

# Precompute KB embeddings
print("Computing knowledge base embeddings...")
kb_embeddings = model.encode(kb_prompts, convert_to_numpy=True, normalize_embeddings=True)
print(f"✅ Computed embeddings for {len(kb_prompts)} knowledge base snippets")

In [ ]:
# =========================
# Graph utilities
# =========================

def _get_type(v):
    d = _record_dictionary(v)
    if not d:
        return None
    try:
        return Dictionary.ValueAtKey(d, "type")
    except Exception:
        return None

def _get_all_by_type(graph, room_type):
    room_type = room_type.strip().lower()
    verts = TGraph.Vertices(graph)
    return [v for v in verts
            if (t := _get_type(v)) and str(t).strip().lower() == room_type]

def _first_by_type(graph, room_type):
    lst = _get_all_by_type(graph, room_type)
    return lst[0] if lst else None

def _new_room_vertex(room_type, area=None, cloud_vertex=0):
    import random
    x = random.sample(range(0,100), 1)[0]
    y = random.sample(range(0,100), 1)[0]
    z = random.sample(range(0,100), 1)[0]
    v = Vertex.ByCoordinates(_as_topology(x), _as_topology(y), _as_topology(z))
    keys = ["type"]
    vals = [room_type]
    if area is not None:
        try:
            vals_area = float(area)
        except Exception:
            vals_area = None
        if vals_area is not None:
            keys.append("area")
            vals.append(vals_area)
    d = Dictionary.ByKeysValues(keys, vals)
    v = Topology.SetDictionary(v, d)
    return v

def _safe_add_vertex(graph, v):
    try:
        TGraph.AddVertex(graph, _as_topology(v))
        return graph
    except Exception as e:
        print(f"  Warning: AddVertex failed: {e}")
        return graph

def _safe_add_edge(graph, sv, ev):
    try:
        e = Edge.ByStartVertexEndVertex(_as_topology(sv), _as_topology(ev), silent=True)
        if e == None:
            sv, ev = Vertex.Separate(_as_topology([sv, ev]), minDistance=1)
            e = Edge.ByStartVertexEndVertex(_as_topology(sv), _as_topology(ev), silent=True)
        TGraph.AddEdge(graph, e)
        return graph
    except Exception as e:
        print(f"  Warning: AddEdge failed: {e}")
        return graph

def _sanitize_list_str(list_str):
    # Parses "'a', 'b', 'c'" into ["a","b","c"]
    return [s.strip() for s in re.findall(r"'([^']+)'", list_str) if s.strip()]

def _parse_area(text):
    m = re.search(r"\barea\s*=\s*([0-9]+(?:\.[0-9]+)?)\b", text, flags=re.IGNORECASE)
    if not m:
        return None
    try:
        return float(m.group(1))
    except Exception:
        return None

def graph_types_and_adjacency(graph):
    """
    Returns:
      types: sorted list of distinct room types in the graph
      adjacency: dict[type] -> set(neighbor_types)
    """
    verts = TGraph.Vertices(graph)
    types = []
    for v in verts:
        t = _get_type(v)
        if t:
            types.append(str(t).strip().lower())
    types = sorted(set(types))

    # Build adjacency via Graph.AdjacentVertices (recommended in your setup)
    adj = {t: set() for t in types}
    for v in verts:
        t = _get_type(v)
        if not t: 
            continue
        t = str(t).strip().lower()
        try:
            nbs = TGraph.AdjacentVertices(graph, v)  # per your earlier preference
        except Exception:
            nbs = []
        for nb in nbs:
            nt = _get_type(nb)
            if nt:
                adj[t].add(str(nt).strip().lower())
    return types, adj

def graph_to_text(graph):
    """
    A compact textual description of the current graph for retrieval + prompting.
    """
    types, adj = graph_types_and_adjacency(graph)
    parts = [f"Current rooms: {', '.join(types) if types else 'none'}."]
    for t in types:
        neigh = sorted(adj.get(t, []))
        if neigh:
            parts.append(f"'{t}' connected to: {', '.join(neigh)}.")
    return " ".join(parts)

In [ ]:
# =========================
# Retrieval helpers
# =========================

def retrieve_kb_context(graph, k=4):
    """
    Use the embedding model to retrieve top-k KB snippets related to the current graph.
    """
    query = graph_to_text(graph)
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    sims = cosine_similarity(q_emb, kb_embeddings)[0]
    idx = np.argsort(-sims)[:k]
    return [kb_prompts[i] for i in idx]

def examples_from_kb(kb_snippets, graph):
    """
    Build few-shot examples shaped exactly like required instruction lines,
    including 'area=<number>' for Add lines.
    """
    types, adj = graph_types_and_adjacency(graph)
    ex = []

    # Example Add instructions based on frequent patterns and area hints
    # Look for gaps in common triads: living room <-> dining room <-> kitchen
    need_kitchen = ("kitchen" not in types) and ("dining room" in types or "living room" in types)
    if need_kitchen:
        area = _area_hint_for("kitchen")
        ex.append(f"Add 'kitchen' connected to 'dining room', 'living room' area={area}")

    need_dining = ("dining room" not in types) and ("kitchen" in types and "living room" in types)
    if need_dining:
        area = _area_hint_for("dining room")
        ex.append(f"Add 'dining room' connected to 'living room', 'kitchen' area={area}")

    # Bedroom/bathroom/hallway pattern
    if "bedroom" in types and "bathroom" not in types:
        area = _area_hint_for("bathroom")
        ex.append(f"Add 'bathroom' connected to 'hallway' area={area}")
    if "bedroom" in types and "hallway" not in types:
        area = _area_hint_for("hallway")
        ex.append(f"Add 'hallway' with no connections area={area}")

    # If nothing to add, consider a benign connect example
    if not ex and "kitchen" in types and "pantry" in types:
        ex.append("Connect 'kitchen' to 'pantry'")

    # Cap examples, ensure uniqueness
    seen = set()
    out = []
    for s in ex:
        if s not in seen:
            out.append(s); seen.add(s)
    # If still empty, seed one generic example to anchor format
    if not out:
        out = ["Add 'storage' with no connections area=4"]

    # Optionally include selected KB lines as comments (LLM sees them, not parsed):
    # We won't include them as instructions to avoid mis-parsing.
    return out

In [ ]:
# =========================
# LLM caller (area enforced, multi-backend)
# =========================

def query_llm(prompt, examples):
    """
    Calls LLM and requests ONLY instruction lines (no explanations).
    Enforces 'area=<number>' for any Add instruction.
    Supports multiple backends including Anthropic Claude.
    """
    system_msg = (
        "You are a graph expansion assistant for residential floorplans.\n\n"
        "Output rules:\n"
        "1) Each line is ONE instruction. Allowed forms:\n"
        "   - Add 'room_type' connected to 'type1', 'type2', ... area=<number>\n"
        "   - Add 'room_type' with no connections area=<number>\n"
        "   - Connect 'type_a' to 'type_b'\n"
        "   - Remove 'room_type'\n"
        "   - STOP\n"
        "2) For every Add instruction, you MUST append 'area=<number>' (square meters).\n"
        "3) Do NOT explain. Output ONLY instruction lines.\n"
    )
    
    user_content = "Examples:\n" + "\n".join(examples) + "\n\n" + prompt

    if BACKEND == "anthropic":
        # Claude doesn't use separate system messages - combine them
        combined_prompt = f"{system_msg}\n\n{user_content}"
        
        response = client.messages.create(
            model=LLM_MODEL,
            max_tokens=1000,
            messages=[
                {"role": "user", "content": combined_prompt}
            ]
        )
        return response.content[0].text.strip()
    
    else:
        # OpenAI-compatible backends (openai, ollama, lmstudio)
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_content},
            ],
        )
        return response.choices[0].message.content.strip()

In [ ]:
# =========================
# Applying suggestions (with area persistence)
# =========================

def apply_suggestion(graph, suggestion):
    """
    Applies LLM suggestions to the graph. Supports:
      - Add 'room_type' connected to 'type1', 'type2', ... area=<number>
      - Add 'room_type' with no connections area=<number>
      - Connect 'type_a' to 'type_b'
      - Remove 'room_type'
      - STOP

    On Add: creates a vertex with dictionary keys:
      - "type": <room_type>
      - "area": <float m^2> (if provided)
    """
    suggestion = (suggestion or "").strip()
    if not suggestion:
        print("  Empty suggestion. Nothing to do.")
        return graph, True

    lines = [ln.strip() for ln in suggestion.splitlines() if ln.strip()]
    stop_signaled = False
    new_graph = graph

    for line in lines:
        l = line.strip()

        # STOP
        if l.lower() == "stop":
            print("  STOP signal detected. Stopping.")
            stop_signaled = True
            continue

        # Add 'X' connected to 'a','b',... area=NN
        m_add_conn = re.match(
            r"^add\s+'([^']+)'\s+connected\s+to\s+(.+?)\s*(area\s*=\s*[0-9]+(?:\.[0-9]+)?)?\s*$",
            l, flags=re.IGNORECASE
        )
        if m_add_conn:
            room_type = m_add_conn.group(1).strip()
            neighbor_str = m_add_conn.group(2)
            area_token = m_add_conn.group(3) or ""
            area_val = _parse_area(area_token) or _parse_area(l)

            v_new = _new_room_vertex(room_type, area_val)
            new_graph = _safe_add_vertex(new_graph, v_new)
            print(f"  Added node '{room_type}'" + (f" with area={area_val} m²" if area_val is not None else ""))

            neighbors = _sanitize_list_str(neighbor_str)
            for nb in neighbors:
                v_target = _first_by_type(new_graph, nb)
                if v_target:
                    new_graph = _safe_add_edge(new_graph, v_new, v_target)
                    print(f"    Connected '{room_type}' to '{nb}'")
                else:
                    print(f"    Warning: No node of type '{nb}' found to connect.")
            continue

        # Add 'X' with no connections area=NN
        m_add_noc = re.match(
            r"^add\s+'([^']+)'\s+with\s+no\s+connections\s*(area\s*=\s*[0-9]+(?:\.[0-9]+)?)?\s*$",
            l, flags=re.IGNORECASE
        )
        if m_add_noc:
            room_type = m_add_noc.group(1).strip()
            area_token = m_add_noc.group(2) or ""
            area_val = _parse_area(area_token) or _parse_area(l)

            v_new = _new_room_vertex(room_type, area_val)
            new_graph = _safe_add_vertex(new_graph, v_new)
            print(f"  Added node '{room_type}'" + (f" with area={area_val} m²" if area_val is not None else ""))
            continue

        # Connect 'a' to 'b'
        m_conn = re.match(r"^connect\s+'([^']+)'\s+to\s+'([^']+)'\s*$", l, flags=re.IGNORECASE)
        if m_conn:
            a = m_conn.group(1).strip()
            b = m_conn.group(2).strip()
            va = _first_by_type(new_graph, a)
            vb = _first_by_type(new_graph, b)
            if va and vb:
                new_graph = _safe_add_edge(new_graph, va, vb)
                print(f"  Connected '{a}' to '{b}'")
            else:
                if not va:
                    print(f"  Warning: No node of type '{a}' found.")
                if not vb:
                    print(f"  Warning: No node of type '{b}' found.")
            continue

        # Remove 'X'
        m_rem = re.match(r"^remove\s+'([^']+)'\s*$", l, flags=re.IGNORECASE)
        if m_rem:
            rtype = m_rem.group(1).strip().lower()
            removed = False
            for v in list(TGraph.Vertices(new_graph)):
                t = _get_type(v)
                if t and str(t).strip().lower() == rtype:
                    new_graph = TGraph.RemoveVertex(new_graph, v)
                    removed = True
                    print(f"  Removed node '{rtype}'")
            if not removed:
                print(f"  Warning: No node of type '{rtype}' to remove.")
            continue

        print(f"  Unrecognized instruction: {l}")

    return new_graph, stop_signaled

In [ ]:
# =========================
# GraphRAG driver functions
# =========================

def build_prompt_for_graph(graph):
    """
    Combine current graph description + retrieved KB into a single prompt
    that asks for next best edits (Add/Connect/Remove/STOP).
    """
    current = graph_to_text(graph)
    kb_ctx = retrieve_kb_context(graph, k=4)
    ex = examples_from_kb(kb_ctx, graph)

    prompt = (
        f"{current}\n\n"
        "Propose 1–3 next instructions to improve completeness and plausibility.\n"
        "Remember: every Add MUST include area=<number> (m²)."
    )
    return prompt, ex

def suggest_next(graph):
    """
    Runs the full GraphRAG step: retrieve KB → craft examples → query LLM
    → return raw suggestions string.
    """
    prompt, ex = build_prompt_for_graph(graph)
    return query_llm(prompt, ex)

# =========================
# Graph initializer
# =========================

def initialize_graph(seed_types=None, cloud_index=0):
    """
    Create a minimal graph from a list of room types (strings).
    Each vertex gets a dictionary with 'type' (and no area yet).
    """
    seed_types = seed_types or ["living room", "kitchen", "bedroom"]
    vertices = []
    for rt in seed_types:
        v = _new_room_vertex(rt, area=None)
        vertices.append(v)
    try:
        # If available in your TopologicPy version:
        g = TGraph.ByVerticesEdges(vertices, [])
    except Exception:
        # Fallback: incrementally add to an empty graph
        try:
            g = TGraph.ByVerticesEdges([], [])
        except Exception:
            # Last resort: attempt to add starting from the first vertex
            g = TGraph.ByVerticesEdges([vertices[0]], [])
            for v in vertices[1:]:
                TGraph.AddVertex(g, _as_topology(v))
            return g
        for v in vertices:
            TGraph.AddVertex(g, _as_topology(v))
    return g

# =========================
# Convenience: one-step expand
# =========================

def expand_once(graph):
    """
    Compute suggestions via GraphRAG+LLM and apply them.
    Returns: (new_graph, stop_flag, suggestions_text)
    """
    suggestions = suggest_next(graph)
    new_graph, stop_flag = apply_suggestion(graph, suggestions)
    return new_graph, stop_flag, suggestions

In [ ]:
# =========================
# Test basic functionality
# =========================

# Test initialization and basic expansion
g = initialize_graph(["living room", "dining room"])
print("Initial graph:", graph_to_text(g))

s = suggest_next(g)
print("LLM suggestions:\n", s)

g2, stop, _ = expand_once(g)
print("\nAfter expansion:", graph_to_text(g2))

# Inspect stored areas
print("\nVertex details:")
for v in TGraph.Vertices(g2):
    d = _record_dictionary(v)
    t = Dictionary.ValueAtKey(d, "type")
    a = Dictionary.ValueAtKey(d, "area")
    print(f"  {t}: area={a} m²")

In [ ]:
# =========================
# Graph prompt generation function  
# =========================

def generate_prompt_from_graph(graph):
    """
    Generates a simple text description of the input graph for use in LLM prompting.
    Vertices are listed by index and label/type (if available), and edges are shown as pairs of indices.

    Parameters
    ----------
    graph : topologic_core.Graph
        The input graph.

    Returns
    -------
    str
        A textual description of the graph's current state.
    """
    from topologicpy.TGraph import TGraph
    from topologicpy.Topology import Topology
    from topologicpy.Dictionary import Dictionary

    vertices = TGraph.Vertices(graph) or []
    edges = _edge_records(graph) or []

    # Build vertex descriptions
    vertex_descs = []
    for i, v in enumerate(vertices):
        d = _record_dictionary(v)
        label = None
        if d:
            label = Dictionary.ValueAtKey(d, "type") or Dictionary.ValueAtKey(d, "label")
        if not label:
            label = f"vertex_{i}"
        vertex_descs.append(f"{i}:{label}")

    # Build edge descriptions
    edge_descs = []
    for e in edges:
        sv = Edge.StartVertex(_as_topology(e))
        ev = Edge.EndVertex(_as_topology(e))
        try:
            si = vertices.index(sv)
            ei = vertices.index(ev)
            edge_descs.append(f"[{si},{ei}]")
        except ValueError:
            continue

    return "Vertices: " + ", ".join(vertex_descs) + "; Edges: " + ", ".join(edge_descs)

# Helper function for averaging listed nodes
def average_listed_nodes(known_graphs):
    total = sum(len(pairs) for name, pairs in known_graphs)
    return total / len(known_graphs) if known_graphs else 1

### STEP 5: Main Loop with Claude Integration

In [ ]:
# =========================
# Main iterative expansion loop
# =========================

def expand_graph_iterative(
    graph,
    user_prompt: str,
    *,
    known_graphs,
    kb_embeddings,
    kb_prompts,
    model,  # must expose .encode(str) -> vector
    query_llm,  # callable(prompt:str, examples:list[str]) -> str
    apply_suggestion,  # callable(graph, suggestion) -> (graph, stop:bool)
    generate_prompt_from_graph,  # callable(graph) -> str
    average_listed_nodes,  # callable(known_graphs) -> float
    max_steps_multiplier: int = 8,
    retrieval_top_k: int = 3,
    verbose: bool = True
):
    """
    Re-integrated graph expansion loop (STATELESS, one-suggestion per turn).
    Returns (final_graph, intermediate_graphs).
    """
    # Compute max steps from corpus stats (your original heuristic)
    max_steps = math.ceil(average_listed_nodes(known_graphs) * max_steps_multiplier)
    if verbose:
        print("Maximum Number of Suggestions:", max_steps)
        print(f"Using {BACKEND} backend with model: {LLM_MODEL}")

    inter_graphs = []
    stop = False
    step = 0

    while not stop and step < max_steps:
        base_prompt = generate_prompt_from_graph(graph)
        if verbose:
            print("Base Prompt:", base_prompt)
        
        prompt = (
            "STATELESS: On each turn, IGNORE previous replies and plans and decide ONLY from the current user message.\n"
            "No commentary, no code blocks. Starting from " + base_prompt +
            ", expand the graph of this house by suggesting one additional node. "
            "If you think the graph does not need additional nodes, then just return the same graph and say STOP. "
            "The graph should have an entrance, a kitchen, a dining room, a living room, and EXACTLY " + user_prompt
        )

        # Retrieve few-shot examples using embedding similarity
        query_embedding = model.encode(prompt)
        sims = cosine_similarity([query_embedding], kb_embeddings)[0]
        top_indices = np.argsort(sims)[-retrieval_top_k:][::-1]
        retrieved_examples = [kb_prompts[i] for i in top_indices]

        # Query LLM
        suggestion = query_llm(prompt, retrieved_examples) or ""
        step_str = str(step + 1).zfill(2)
        if verbose:
            print(f"{step_str}: {suggestion}")

        # Apply suggestion
        graph, stop = apply_suggestion(graph, suggestion)
        inter_graphs.append(graph)
        step += 1

        if step == max_steps and verbose:
            print("Maximum number of steps reached. Stopping.")

    # Cleanup pass (your original line)
    from topologicpy.TGraph import TGraph  # local import to avoid circulars if any
    graph = TGraph.RemoveIsolatedVertices(graph)
    return graph, inter_graphs

# Execute the main expansion
print(f"Starting graph expansion with {BACKEND} backend...")
graph = initialize_graph(["entrance"])

graph, inter_graphs = expand_graph_iterative(
    graph=graph,
    user_prompt=user_prompt,
    known_graphs=known_graphs,
    kb_embeddings=kb_embeddings,
    kb_prompts=kb_prompts,
    model=model,
    query_llm=query_llm,
    apply_suggestion=apply_suggestion,
    generate_prompt_from_graph=generate_prompt_from_graph,
    average_listed_nodes=average_listed_nodes
)

### STEP 6: Display the Resulting Graph Adjacency Matrix Figure

In [ ]:
from topologicpy.Plotly import Plotly
from topologicpy.TGraph import TGraph
fig = TGraph.AdjacencyMatrixFigure(graph, title="My Adjacency Matrix - Claude Generated", reverse=True, fontColor="black", vertexKey="type", showZero=False, fontSize=11, backgroundColor="red", diagonalColor="yellow")
Plotly.Show(fig)

In [ ]:
# Draw the flat graph with color-coding
from topologicpy.Wire import Wire

flat_g = TGraph.Reshape(graph)
room_type_to_color = {
    "bedroom": "lightblue",
    "bedroom1": "lightblue",
    "bedroom2": "lightblue",
    "bedroom3": "lightblue",
    "bedroom4": "lightblue",
    "bedroom5": "lightblue",
    "master bedroom": "lightblue",
    "main bedroom": "lightblue",
    "guest bedroom": "lightskyblue",
    "bathroom": "lightgray",
    "bathroom1": "lightgray",
    "bathroom2": "lightgray",
    "bathroom3": "lightgray",
    "bathroom4": "lightgray",
    "bathroom5": "lightgray",
    "toilet": "lightgray",
    "toilet1": "lightgray",
    "toilet2": "lightgray",
    "toilet3": "lightgray",
    "toilet4": "lightgray",
    "closet": "saddlebrown",
    "closet1": "saddlebrown",
    "closet2": "saddlebrown",
    "closet3": "saddlebrown",
    "closet4": "saddlebrown",
    "closet5": "saddlebrown",
    "storage": "saddlebrown",
    "storage room": "saddlebrown",
    "storageroom": "saddlebrown",
    "kitchen": "peachpuff",
    "diningroom": "khaki",
    "dining room": "khaki",
    "pantry": "linen",
    "livingroom": "mediumseagreen",
    "living room": "mediumseagreen",
    "hallway": "darkslategray",
    "entrance": "gold",
    "library": "plum",
    "laundryroom": "powderblue",
    "laundry room": "powderblue",
    "garage": "slategray",
    "basement": "dimgray",
    "homeoffice": "lightsteelblue",
    "home office": "lightsteelblue",
    "study": "lightsteelblue",
    "study room": "lightsteelblue",
    "studyroom": "lightsteelblue",
    "mudroom": "burlywood",
    "mud room": "burlywood",
    "garden": "lightgreen",
    "backyard": "lightgreen",
    "back yard": "lightgreen",
    "frontyard": "lightgreen",
    "front yard": "lightgreen",
    "attic": "brown",
    "game room": "orange",
    "gameroom": "orange"
}

vertices = TGraph.Vertices(flat_g)
_ = TGraph.DegreeCentrality(flat_g)

for v in vertices:
    d = _record_dictionary(v)
    room_color = room_type_to_color.get(Dictionary.ValueAtKey(d, "type"), "black")
    d = Dictionary.SetValueAtKey(d, "room_color", room_color)
    d = Dictionary.SetValueAtKey(d, "degree_centrality", Dictionary.ValueAtKey(d, "degree_centrality")*4 + 4) #exaggerate the size of the vertices based on their degree.
    v = Topology.SetDictionary(v, d)

br = Wire.BoundingRectangle(_as_topology(TGraph.Topology(flat_g)))
br = Topology.Scale(_as_topology(br), origin=Topology.Centroid(_as_topology(br)), x=1.5, y=1.5, z=1)
Topology.Show(br, flat_g, vertexLabelKey="type", vertexSize=0.001, vertexSizeKey="degree_centrality", vertexColorKey="room_color", showVertexLabel=True, projection="orthographic", camera=[0,0,1])

### STEP 7: Save the PNG images for the Animation of the Growth of the Graph.
You can use ezgif.com to assemble them into an animation

In [ ]:
from topologicpy.Wire import Wire
from topologicpy.Plotly import Plotly

print(f"Generating {len(inter_graphs)} animation frames...")
final_figures = []

for i, g in enumerate(inter_graphs):
    print(f"Processing frame {i+1}/{len(inter_graphs)}")
    new_g = TGraph.Intersect(flat_g, g)
    _ = TGraph.DegreeCentrality(new_g)
    vertices = TGraph.Vertices(new_g)
    
    for v in vertices:
        d = _record_dictionary(v)
        room_color = room_type_to_color.get(Dictionary.ValueAtKey(d, "type"), "black")
        d = Dictionary.SetValueAtKey(d, "room_color", room_color)
        d = Dictionary.SetValueAtKey(d, "degree_centrality", Dictionary.ValueAtKey(d, "degree_centrality")*4 + 4) #exaggerate the size of the vertices based on their degree.
        v = Topology.SetDictionary(v, d)
    
    fig = Topology.Show(br, new_g, backgroundColor="white",
                      vertexLabelKey="type",
                      vertexSize=1,
                      vertexSizeKey="degree_centrality",
                      vertexColorKey="room_color",
                      showVertexLabel=True,
                      projection="orthographic",
                      camera = [0,0,1],
                      showFigure = False)
    final_figures.append(fig)

print(f"Saving {len(final_figures)} PNG files...")
for i, fig in enumerate(final_figures):
    path = folder+"/claude_figure_"+(str(i+1).zfill(2))+".png"
    try:
        status = Plotly.ExportToImage(fig, path, format="png", width="800", height="700")
        print(f"Saved: {path}")
    except Exception as e:
        print(f"Failed to save {path}: {e}")

print("\n=== CLAUDE GRAPHRAG GENERATION COMPLETE ===\n")
print(f"Backend: {BACKEND}")
print(f"Model: {LLM_MODEL}")
print(f"Final graph has {len(TGraph.Vertices(graph))} rooms")
print(f"Generated {len(inter_graphs)} intermediate graphs")
print(f"Animation frames saved to: {folder}")

In [ ]:
# Check what methods are available in the Graph class
import sys
print("Python path:")
for path in sys.path:
    if 'topologic' in path.lower():
        print(f"  {path}")

print("\nTrying to import Graph and check its methods...")
try:
    from topologicpy.TGraph import TGraph
    print("Graph class imported successfully")
    
    # Check available methods
    methods = [method for method in dir(TGraph) if not method.startswith('_')]
    print(f"\nAvailable methods in Graph class ({len(methods)} total):")
    
    # Look for adjacency-related methods
    adjacency_methods = [m for m in methods if 'adjacency' in m.lower() or 'matrix' in m.lower()]
    print(f"\nAdjacency/Matrix related methods: {adjacency_methods}")
    
    # Look for figure/plot related methods
    figure_methods = [m for m in methods if 'figure' in m.lower() or 'plot' in m.lower()]
    print(f"Figure/Plot related methods: {figure_methods}")
    
except Exception as e:
    print(f"Error importing Graph: {e}")

In [ ]:
# Let's check what methods are available on the Graph class
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

from topologicpy.TGraph import TGraph
print("Available methods in Graph class:")
methods = [method for method in dir(TGraph) if not method.startswith('_')]
for method in sorted(methods):
    print(f"  {method}")